In [11]:
# Make sure to import packages and run code from the initial_setup notebook.

# This code is setup to run leave-one-out cross-validation for Function 1. To use it for other functions,
# add X values. For example, for Function 5, the X values would be:
# X_1 = df_cv['X_1'].values
# X_2 = df_cv['X_2'].values
# X_3 = df_cv['X_3'].values
# X_4 = df_cv['X_4'].values

df_cv = df_function_1

X_1 = df_cv['X_1'].values
X_2 = df_cv['X_2'].values
y = df_cv['y'].values

# Aggressive output scaling plus StandardScaler for Function 1. For Functions 2-8, I used the yeo-johnson power transformation below as recommended by HEBO. 
# When running Functions 2-8, it's recommended to comment out this aggressive scaling as it will negatively impact your results.
alpha = 0.02
y_eng = np.sign(y) * np.power(np.abs(y), alpha)

scaler = StandardScaler()
y_eng = scaler.fit_transform(y_eng.reshape(-1, 1)).flatten()

# Scale y values for Functions 2-8. 
# pt = PowerTransformer(method='yeo-johnson')
# y_eng = pt.fit_transform(y.reshape(-1, 1))

# Add additional X values if running other functions, similar to the above. For example, running this on Function 5
# would look like points = np.column_stack((X_1, X_2, X_3, X_4))
points = np.column_stack((X_1, X_2))

# Grid search on smoothing levels.
smoothing = [0.0, 1e-6, 1e-4, 0.01, 0.1]

# Kernels of the RBFInterpolator where no epsilon value is allowed.
grid_no_epsilon = {
    'kernel': ['linear', 'thin_plate_spline', 'cubic', 'quintic'],
    'smoothing': smoothing
}

# Kernels of the RBFInterpolator where epsilon value is required.
grid_with_epsilon = {
    'kernel': ['multiquadric', 'inverse_multiquadric', 'inverse_quadratic', 'gaussian'],
    'epsilon': [0.01, 0.1, 0.5, 1.0, 2.0, 3.0, 6.0, 10.0, 14.0, 18.0], 
    'smoothing': smoothing
}

# Generate combinations for grid search.
def get_combinations(grid):
    keys = grid.keys()
    values = grid.values()
    return [dict(zip(keys, combo)) for combo in itertools.product(*values)]

all_configs = get_combinations(grid_no_epsilon) + get_combinations(grid_with_epsilon)

# LeaveOneOut cross-validation for grid search.
loo = LeaveOneOut()

best_score = float('inf')
best_config = None
results = []

# Loop through all configurations.
for config in all_configs:
    fold_scores = []
    
    for train_index, test_index in loo.split(points):
        X_train, X_test = points[train_index], points[test_index]
        y_train, y_test = y_eng[train_index], y_eng[test_index]
        
        try:
            # Initialize interpolator with training data and current configuration
            interpolator = RBFInterpolator(X_train, y_train, **config)
            
            # Predict on the single test point
            y_pred = interpolator(X_test)
            
            # Calculate absolute error for this point
            error = np.abs(y_test[0] - y_pred[0])
            fold_scores.append(error)
            
        except Exception as e:
            # Penalize configurations that crash 
            fold_scores.append(float('inf'))
            break
            
    # Calculate Mean Absolute Error across all LOO splits
    avg_error = np.mean(fold_scores)
    results.append({'config': config, 'error': avg_error})
    
    if avg_error < best_score:
        best_score = avg_error
        best_config = config

print('Top 20 Configurations:')
results_sorted = sorted(results, key=lambda x: x['error'])
for i in range(min(20, len(results_sorted))):
    cfg = results_sorted[i]['config']
    print(f"{i+1}. Error: {results_sorted[i]['error']:.8f} | {cfg}")

Top 20 Configurations:
1. Error: 0.47327395 | {'kernel': 'gaussian', 'epsilon': 14.0, 'smoothing': 0.1}
2. Error: 0.47352384 | {'kernel': 'gaussian', 'epsilon': 18.0, 'smoothing': 0.1}
3. Error: 0.48683444 | {'kernel': 'gaussian', 'epsilon': 14.0, 'smoothing': 0.01}
4. Error: 0.48994064 | {'kernel': 'gaussian', 'epsilon': 18.0, 'smoothing': 0.0}
5. Error: 0.49014090 | {'kernel': 'gaussian', 'epsilon': 18.0, 'smoothing': 1e-06}
6. Error: 0.49108974 | {'kernel': 'gaussian', 'epsilon': 18.0, 'smoothing': 0.01}
7. Error: 0.49201791 | {'kernel': 'gaussian', 'epsilon': 14.0, 'smoothing': 0.0}
8. Error: 0.49277770 | {'kernel': 'gaussian', 'epsilon': 14.0, 'smoothing': 1e-06}
9. Error: 0.49720840 | {'kernel': 'gaussian', 'epsilon': 18.0, 'smoothing': 0.0001}
10. Error: 0.50393753 | {'kernel': 'gaussian', 'epsilon': 14.0, 'smoothing': 0.0001}
11. Error: 0.50788324 | {'kernel': 'inverse_quadratic', 'epsilon': 18.0, 'smoothing': 0.1}
12. Error: 0.51894715 | {'kernel': 'inverse_quadratic', 'epsilo